# NVIDIA Data Center Q4 季度表与表级哈希修复

固定 42 份原始表读取，仅判年度附件的 Q4/Q3 季度表并补全已接受行哈希，不读取 SOXX 结果。


In [1]:
import os,sys,json,hashlib,re,datetime,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_nvda_datacenter_q4_table_lineage_repair_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_HTTP_GETs','parent_v1_Recorder_loads_max','market_outcome_reads','model_fits','native_accounts'))
v1=root/src['parent_v1_card'];assert sha(v1/'plan.json')==src['parent_v1_plan_sha256'] and sha(v1/'result.json')==src['parent_v1_result_sha256'] and sha(v1/'summary.json')==src['parent_v1_summary_sha256']
sum1=json.loads((v1/'summary.json').read_text());assert sum1['recorder_id']==src['parent_v1_recorder_id'] and sum1['nas_evidence']['sha256']==src['parent_v1_NAS_sha256']
assert sha(root/src['Qlib_provider']/'qlib.yaml')==src['Qlib_config_sha256']
prior=json.loads((v1/'result.json').read_text());events=[dict(x) for x in prior['events']];assert len(events)==63
amb={x['accession']:x for x in plan['exact_10_ambiguous_Q4']};assert len(amb)==10 and {e['accession'] for e in events if e['status']=='ambiguous_multiple_Data_Center_rows'}==set(amb)
state={'plan_sha256':sha(card/'plan.json'),'raw_original_reads':0,'raw_original_reads_max':42,'new_Qlib_Recorders':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};state_path=card/'run_state.json';state_path.write_text(json.dumps(state,indent=2)+'\n')
def parse_num(x):
 m=re.fullmatch(r'\$?\s*([0-9][0-9,]*)',x.strip());return int(m.group(1).replace(',','')) if m else None
def parse_row(cells,table_text):
 unit_ok=bool(re.search(r'\$\s*in\s*millions|\(\$\s*in\s*millions\)',table_text,re.I))
 nums=[parse_num(v) for v in cells[1:]];numeric=[x for x in nums if x is not None]
 if not unit_ok or len(numeric)<3 or min(numeric[:3])<=0:return None
 cur,priorq,priory=numeric[:3];growth=cur/priory-1
 pct=[(m.group(1),float(m.group(2))) for m in re.finditer(r'(?:(Up|Down)\s*)?([0-9]+(?:\.[0-9]+)?)\s*%',' '.join(cells[4:]),re.I)]
 reported=(pct[-1][1]*(-1 if pct[-1][0] and pct[-1][0].lower()=='down' else 1)) if pct else None
 diff=abs(100*growth-reported) if reported is not None else None
 if diff is not None and diff>1.5:return None
 return {'current_million':cur,'prior_quarter_million':priorq,'prior_year_million':priory,'same_release_YoY':growth,'reported_YoY_pct':reported,'reported_YoY_diff_pp':diff,'table_unit':'USD_millions'}
read_accessions=[];resolved=[];unresolved=[]
for e in events:
 if e['status'] not in ('numeric_same_release_table','ambiguous_multiple_Data_Center_rows'):continue
 p=root/src['original_SEC_raw_root']/src['original_raw_template'].format(accession=e['accession']);assert p.is_file() and sha(p)==e['raw_sha256']
 full=p.read_text(errors='replace');state['raw_original_reads']+=1;state_path.write_text(json.dumps(state,indent=2)+'\n');read_accessions.append(e['accession'])
 assert state['raw_original_reads']<=plan['budget']['raw_original_reads_max']
 accepted=re.search(r'<ACCEPTANCE-DATETIME>\s*([0-9]{14})',full,re.I);assert accepted and accepted.group(1)==e['accepted_EDGAR_clock']
 docs=[m.group(1) for m in re.finditer(r'<DOCUMENT>(.*?)</DOCUMENT>',full,re.I|re.S)];cfo=[x for x in docs if re.search(r'<TYPE>\s*EX-99\.2(?:\s|<)',x,re.I)];assert len(cfo)==1
 body=re.search(r'<TEXT>(.*?)</TEXT>',cfo[0],re.I|re.S);assert body;soup=BeautifulSoup(body.group(1),'html.parser');matches=[]
 for tab in soup.find_all('table'):
  tabtext=' '.join(tab.stripped_strings)
  if not re.search(r'Revenue\s+by\s+Market\s+Platform',tabtext,re.I):continue
  for tr in tab.find_all('tr'):
   cells=[' '.join(c.stripped_strings) for c in tr.find_all(['td','th'])]
   if cells and re.fullmatch(r'Data\s*Center|Datacenter',cells[0].strip(),re.I):matches.append((tab,tabtext,cells))
 if e['status']=='numeric_same_release_table':
  assert len(matches)==1,(e['accession'],len(matches))
  tab,tabtext,cells=matches[0];parsed=parse_row(cells,tabtext);assert parsed
  assert cells==e['original_row'] and all(parsed[k]==e[k] for k in ('current_million','prior_quarter_million','prior_year_million'))
  assert abs(parsed['same_release_YoY']-e['same_release_YoY'])<1e-12
  assert e['filing_date']==str(datetime.datetime.strptime(e['accepted_EDGAR_clock'][:8],'%Y%m%d').date())
 else:
  quarters=[]
  for tab,tabtext,cells in matches:
   before=tabtext[:tabtext.lower().find(cells[0].lower())]
   if re.search(r'Q4\s*FY[0-9]+',before,re.I) and re.search(r'Q3\s*FY[0-9]+',before,re.I):quarters.append((tab,tabtext,cells))
  if len(quarters)!=1:
   e['q4_repair_status']='not_unique_Q4_Q3_quarter_table';e['q4_candidate_table_count']=len(quarters);unresolved.append(e['accession']);continue
  tab,tabtext,cells=quarters[0];parsed=parse_row(cells,tabtext)
  if parsed is None:
   e['q4_repair_status']='quarter_table_numeric_or_YoY_unqualified';unresolved.append(e['accession']);continue
  e.update(parsed);e['status']='numeric_same_release_table';e['original_row']=cells;e['q4_repair_status']='accepted_original_Q4_quarter_table';resolved.append(e['accession'])
 e['original_row_sha256']=hashlib.sha256(json.dumps(cells,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
 e['enclosing_table_canonical_sha256']=hashlib.sha256(str(tab).encode()).hexdigest()
 e['table_header_signature']=tabtext[:tabtext.lower().find(cells[0].lower())].strip()[-170:]
assert state['raw_original_reads']==42 and len(read_accessions)==42 and len(set(read_accessions))==42
assert sum(e['status']=='numeric_same_release_table' for e in events)==32+len(resolved)
assert all(e.get('original_row_sha256') and e.get('enclosing_table_canonical_sha256') for e in events if e['status']=='numeric_same_release_table')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-09',freq='day'));assert len(cal)==3966
events_by_date=sorted(events,key=lambda r:r['first_conservative_Qlib_signal_date']);feature=[];ages=[];accessions=[];j=0;latest=None
for d in cal:
 while j<len(events_by_date) and pd.Timestamp(events_by_date[j]['first_conservative_Qlib_signal_date'])<=d:latest=events_by_date[j];j+=1
 feature.append(latest['same_release_YoY'] if latest and latest['status']=='numeric_same_release_table' else np.nan)
 ages.append((d-pd.Timestamp(latest['first_conservative_Qlib_signal_date'])).days if latest else np.nan)
 accessions.append(latest['accession'] if latest else None)
frame=pd.DataFrame({'signal_date':cal,'NVDA_Data_Center_same_release_YoY':feature,'source_age_calendar_days':ages,'source_accession':accessions}).set_index('signal_date')
train=frame.loc['2011-01-03':'2022-12-30'];held=frame.loc['2023-01-03':'2026-10-09'];assert len(train)==3020 and len(held)==946
old_events={e['accession']:e for e in prior['events']}
for e in events:
 old=old_events[e['accession']]
 if old['status']=='numeric_same_release_table':assert e['status']==old['status'] and e['same_release_YoY']==old['same_release_YoY'] and e['original_row']==old['original_row']
 if old['status']=='no_exact_Data_Center_market_platform_row':assert e==old
report={'decision':'NVDA_original_Data_Center_Q4_quarter_table_and_hash_lineage_repaired_source_only','plan_sha256':sha(card/'plan.json'),'parent_v1_result_sha256':src['parent_v1_result_sha256'],'raw_original_reads':state['raw_original_reads'],'accepted_Q4_repairs':len(resolved),'unresolved_Q4':len(unresolved),'resolved_Q4_accessions':resolved,'unresolved_Q4_accessions':unresolved,'original32_values_and_rows_unchanged':True,'numeric_events':sum(e['status']=='numeric_same_release_table' for e in events),'numeric_rows_with_row_and_table_hashes':sum(bool(e.get('original_row_sha256') and e.get('enclosing_table_canonical_sha256')) for e in events if e['status']=='numeric_same_release_table'),'train_known_days':int(train.NVDA_Data_Center_same_release_YoY.notna().sum()),'held_known_days':int(held.NVDA_Data_Center_same_release_YoY.notna().sum()),'train_rows':len(train),'held_rows':len(held),'events':events,'historical_first_issuer_release_qualified':False,'platform_definition_continuity_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
static=pd.concat({'feature':frame[['NVDA_Data_Center_same_release_YoY']],'meta':frame[['source_age_calendar_days']]},axis=1);static.index=pd.MultiIndex.from_arrays([cal,['NVDA_DC']*len(cal)],names=['datetime','instrument']);ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-09')});assert len(ds.prepare('train',col_set=['feature','meta']))==3020 and len(ds.prepare('held',col_set=['feature','meta']))==946
tracking=card/'qlib_recorder';assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds);rec.save_objects(artifact_path='analysis',event_rows=events,source_frame=frame,report={k:v for k,v in report.items() if k!='events'});rec.log_metrics(numeric_events=report['numeric_events'],q4_repairs=len(resolved),train_known=report['train_known_days'],held_known=report['held_known_days'],model_fits=0,native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':report['plan_sha256'],'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'result':'result.json','model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Quarterly Q4 table selection and canonical table hashes repair only source lineage, not original issuer first-public timing.','2016-2020 and later platform definition changes need separate semantics review before a predictive factor.','No market outcome, information fit, fee-net account or independent forward evidence.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='nvda-dc-q4-repair-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert proc.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='finished';state['new_Qlib_Recorders']=1;state_path.write_text(json.dumps(state,indent=2)+'\n')
print(json.dumps({'resolved_Q4':len(resolved),'unresolved':unresolved,'numeric_events':report['numeric_events'],'train_known':report['train_known_days'],'held_known':report['held_known_days'],'recorder_id':rid},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[29601:MainThread](2026-10-23 08:19:00,570) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[29601:MainThread](2026-10-23 08:19:00,571) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[29601:MainThread](2026-10-23 08:19:00,572) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


[29601:MainThread](2026-10-23 08:19:00,598) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[29601:MainThread](2026-10-23 08:19:00,606) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_nvda_datacenter_q4_table_lineage_repair_20261023.


[29601:MainThread](2026-10-23 08:19:00,607) INFO - qlib.workflow - [exp.py:258] - Experiment 837375431291646527 starts running ...


[29601:MainThread](2026-10-23 08:19:00,689) INFO - qlib.workflow - [recorder.py:345] - Recorder fc47f165d79b4caaaf5e6759f4dd86b7 starts running under Experiment 837375431291646527 ...


[29601:MainThread](2026-10-23 08:19:01,469) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "resolved_Q4": 10,
  "unresolved": [],
  "numeric_events": 42,
  "train_known": 1671,
  "held_known": 946,
  "recorder_id": "fc47f165d79b4caaaf5e6759f4dd86b7"
}


# NVIDIA Data Center Q4 季度表与来源哈希修复

仅对 v1 中 **10 份**年度 Q4 重名行，使用同一 `EX-99.2` 原件中 `Revenue by Market Platform` 表头的 `Q4 FY` 当前季与 `Q3 FY` 前季列判定季度表，排除只列 `FY` 的全年表；同时重读原已接受的 **32 份**原件核对单元格与数值不变，并给全部接受行补上规范化单元格行哈希、所在 HTML 表哈希及原 SEC 文件哈希。实际原件读取 **42/42**，0 新 HTTP 请求、0 SOXX 收益读取、0 模型拟合、0 账户。

10 份 Q4 原件均唯一选出季度表，原 32 个已接受当前季/前季/上年同期值与原行逐一保持不变。数值事件增至 **42/63**，三层来源身份在所有 42 个事件上齐全；其余 21 个早期“无精确平台行”仍未知，没有从较晚公告回填。按 SEC 接收日期的下一真实 Qlib 交易日保守映射，2011–2022 训练 **1,671/3,020** 日有值，2023–2026 复用评价 **946/946** 日有值。

原生 Qlib Recorder `fc47f165d79b4caaaf5e6759f4dd86b7` 已读回 63 个事件、3,966 个日历框架行和 3,020/946 行 DatasetH；Yanxuan NAS 归档的远端哈希及限定恢复通过。这个修复只解决季度与全年表歧义及 v1 的行/表哈希遗漏。Data Center 平台业务范围在收购与披露分类变化前后并非已认证的恒定经济口径，SEC 接收后的下一交易日也不等于已证明的发行人首次公开时刻。

**决定：完整复用评价期的来源数值与表级身份已具备，但首次公开时点和跨业务定义语义仍需独立审查；不把 42 个季度当作 946 个独立事件，也不直接进入费用账户。** 下一步固定定义变化处理和原 Stage1 39 项的匹配信息门；任何通过仍须完整 Stage1 0bp Tiered 自有账户及 SOXX 合同 v2 验收。
